# The Transformer Architecture
> From the seminal paper [*Attention Is All You Need*](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)

Written by Charreau Bell for DS 5690  
Code Version Effective: September 8, 2026

In [ ]:
# Eventually you're going to have to run some code somewhere. It might be best if you run this on Google Colab. If you do that, you need to install these packages, so uncomment the lines below and run them.
# !pip install transformers torch

## Try It Yourself!

First, we're going to take a few minutes for you to read through some parts of the paper on your own. Focus on sections 1-4, with a particular focus on sections 3 and 4. You'll answer two questions on Top Hat for 2 difference sections:

* What is something you learned from this section?
* What is something that is still unclear?

You'll answer this for (1) the Abstract + Sections 1-2, and (2) sections 3 and 4.

You'll have 10 minutes.

Submit both responses through **Top Hat**, which is linked on Brightspace.

## Introductory Sections

Abstract, Introduction, and Background

### Abstract: What does the paper claim?

<img src="assets/paper-01-abstract.png" alt="The complete highlighted abstract from Attention Is All You Need" style="max-height:500px; width:auto; display:block; margin:auto;">

### Introduction: What motivated the creation of the transformer architecture?

<img src="assets/paper-02a-introduction-problem.png" alt="Introduction paragraph describing recurrent computation" style="max-height:285px; width:auto; display:block; margin:auto;">
<br>
<img src="assets/paper-02b-introduction-proposal.png" alt="Final Introduction paragraphs presenting the Transformer" style="max-height:300px; width:auto; display:block; margin:auto;">

### Background: What was going on in the field of NLP in 2017?

<img src="assets/paper-03-background.png" alt="Highlighted Section 2 background" style="max-height:600px; width:auto; display:block; margin:auto;">

## A Brief Aside: NLP Tasks and Smaller Models

<div style="width:100%; display:flex; table-layout:fixed;">
  <div style="align-content:center; width:30%;">
    Historically, NLP research has focused on solving <em>specific</em> NLP tasks rather than building a general-purpose model, as this was not something considered possible at the time. This was largely due to:<br>
    <ul>
      <li>Lack of available data</li>
      <li>Lack of compute</li>
      <li>Lack of algorithmic sophistication</li>
    </ul>
    As a result, NLP research contained many named tasks and specialized models rather than one general chat interface. Hugging Face pipelines make these different input and output contracts visible.
  </div>
  <div style="vertical-align:center; width:5%;">
    <br>
  </div>
  <div style="vertical-align:top; width:65%;">
    <img src="assets/hf-nlp-tasks.png" alt="NLP Tasks" style="max-height:500px; width:auto; margin-top: 0;">
  </div>
</div>

Above, you see the tasks available for NLP, but there are many more depending on the application. You can see [an array of different tasks](https://huggingface.co/tasks) and a plethora of fulfilling models on Hugging Face.

<div style="border-left: 4px solid #1976D2; background: #e3f2fd; padding: 0.5em 1em 0.5em 1.2em; margin: 1em 0; border-radius: 5px; max-width:50ch;">
  <strong><a href="https://huggingface.co/tasks" target="_blank" rel="noopener">Let's see some other tasks.</a></strong>
</div>

### Transfer Learning: Extensive Pre-Training and Switching Heads

The following diagram is another way to think about the blocks or main arrangements of the transformer architecture.

<div style="width:100%; display:flex; table-layout:fixed;">
  <div style="width:65%; min-width:30ch;">
    <img src="assets/transformer-and-head.svg" alt="A transformer body connected to task-specific heads" style="max-height:650px; width:auto; margin-top:0;">
  </div>
  <div style="width:38%; min-width:15ch; max-width:33ch; margin-right:2%;align-content:center;">
    <strong>Many of the tasks that we saw above:</strong>
    <ul style="margin-top:0.5em;">
      <li>Fill in a missing token</li>
      <li>Classify a sentence</li>
      <li>Label each token</li>
      <li>Answer a question from a passage</li>
      <li>Translate one sequence into another</li>
    </ul>
    <p>
      They can be viewed as differing from each other only in the head attached to the transformer body.<br><br>
      Consider a model which is extensively trained and knows foundational relationships, concepts, and meaning. Switching out the model head allows a substantially smaller amount of training to be done to adapt the model to a new task.
    </p>
  </div>
</div>

[Hugging Face LLM Course, Chapter 2](https://huggingface.co/learn/llm-course/chapter2/2)

### Switching out Model Heads
Let's explore this with a simple encoder-based BERT model. Recall that BERT is a pre-trained model that can be used for a variety of NLP tasks, and is generally considered to be encoder-based. Below, you see:
* Different tasks that can be performed with BERT
* The same BERT-base architecture, but with different task heads attached
* Separate model checkpoints that were **fine-tuned** for each task from a BERT-base starting point

Let's look at this content.


In [ ]:
from transformers import pipeline

#### Exploring Different Models and Different Tasks

Change its example, run it, and inspect the kind of output it produces.

In [ ]:
# Option 1: fill a masked token
fill_mask = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased"
)

fill_mask("Attention is all you [MASK].")

In [ ]:
# look at the content of model
fill_mask.model

# specifically look at the decoder
#fill_mask.model.cls.predictions??

In [ ]:
# Option 2: label named entities
ner = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

ner("Ashish Vaswani worked at Google Brain.")

In [ ]:
# look at the content of the model
ner.model

# specifically look at the classifier
#ner.model.classifier??

In [ ]:
# Option 3: answer from a supplied passage
question_answerer = pipeline(
    "question-answering",
    model="deepset/bert-base-cased-squad2"
)

question_answerer(
    question="What architecture did the paper propose?",
    context="The authors proposed the Transformer, an attention-only architecture."
)

In [ ]:
question_answerer.model

# Look at the content of the model
#question_answerer.model??

In [ ]:
from transformers import AutoModel, AutoModelForTokenClassification

bert = AutoModel.from_pretrained("google-bert/bert-base-uncased")
bert

In [ ]:
bert_ner = AutoModelForTokenClassification.from_pretrained(
    "dslim/bert-base-NER"
)

bert_ner.classifier

In [ ]:
print(type(bert))
print(type(bert_ner.bert))
print(type(bert_ner.classifier))

Let's have a final look at a standard decoder-only model.

In [ ]:
from transformers import GPT2Model

gpt2 = GPT2Model.from_pretrained("openai-community/gpt2")
gpt2

In [ ]:
# Look at one repeated decoder block
gpt2.h[0]

## Back to the Paper: The Model Architecture - Section 3

The nomenclature of **encoder** and **decoder** is an old one, related to the behavior and purpose of the encoder and decoder.
<img src="assets/paper-05a-model-definition.png" alt="The paper's formal description of encoder and decoder" style="max-height:150px; width:auto; display:block; margin:auto;">

Below, we see the big picture of the Transformer architecture in its encoder-decoder formulation for machine translation.

<div style="width:100%; display:flex; table-layout:fixed;">
  <div style="width:30%; min-width:30ch;">
    <img src="assets/paper-04-transformer-architecture.png" alt="Figure 1: The Transformer architecture" style="max-height:720px; width:auto;">
  </div>
  <div style="width:70%; min-width:15ch; margin-left:5%;align-content:center;">
    <img src="assets/paper-05b-encoder-decoder-stacks.png" alt="The complete Section 3.1 encoder and decoder paragraphs" style="max-height:400px; width:auto;">
  </div>
</div>


Let's look at a small encoder-decoder model - T5, which was absolutely groundbreaking when it was released. You can learn more about the model [here](https://huggingface.co/google-t5/t5-small).

In [ ]:
# T5-small: a 60.5M-parameter encoder-decoder Transformer
from transformers import T5ForConditionalGeneration

t5 = T5ForConditionalGeneration.from_pretrained("google-t5/t5-small")
t5

### Attention

Attention allows every position directly examine every other relevant position using attention. That removes the sequential-computation bottleneck of RNNs and gives much shorter paths between distant pieces of information. Queries are search requests, other tokens have keys, reflecting the information they contain. The query is compared with keys to obtain weights to obtain relevance scores; after softmax, those scores are turned to weights, attention takes a weighted blend of the values.

<img src="assets/paper-06-attention-definition-pt1.png" alt="The paper's definition of attention" style="width:900px; max-width:100%; height:auto; display:block; margin:0 auto 0;">
<img src="assets/paper-06-attention-definition-pt2.png" alt="The paper's definition of attention" style="width:900px; max-width:100%; height:auto; display:block; margin:0 auto;">

#### Scaled dot-product attention

<div style="width:100%; display:flex; table-layout:fixed; text-align:center;">
  <div style="width:70%;">
    <img src="assets/paper-07a-scaled-attention-text.png" alt="Section 3.2.1 text" style="max-height:550px; width:auto;">
  </div>
  <div style="width:30%; margin-left:2%; align-content:center;">
    <img src="assets/paper-07b-scaled-attention-equation.png" alt="Equation 1 and scaled dot-product attention diagram" style="max-height:600px; width:auto;">
  </div>
</div>

### Multi-head Attention

<div style="display: table; width: 100%; margin-bottom: 24px;">
  <div style="display: table-row;">
    <div style="display: table-cell; width: 70%; vertical-align: top; padding-right: 20px;">
      <img src="assets/paper-08a-multi-head-text-pt1.png" alt="Section 3.2.2 text" style="width: 72%; height: auto; display: block; margin: auto;">
      <img src="assets/paper-08a-multi-head-text-pt2.png" alt="Section 3.2.2b text" style="width: 72%; height: auto; display: block; margin: 0px auto 0;">
    </div>
    <div style="display: table-cell; width: 30%; vertical-align: center; text-align: center;">
      <img src="assets/paper-08b-multi-head-equations.png" alt="Multi-head attention diagram and equations" style="max-height: 600px; width: auto; display: block; margin: auto;">
    </div>
  </div>
</div>

Let's look at some of the code for attention implementations for GPT-2.

In [ ]:
# GPT-2's projections and head dimensions
gpt2.h[0].attn

#gpt2.h[0].attn??

In [ ]:
print("Hidden size:", gpt2.config.n_embd)
print("Attention heads:", gpt2.config.n_head)
print("Size per head:", gpt2.config.n_embd // gpt2.config.n_head)

<div style="border-left: 4px solid rgb(244, 86, 178); background:rgb(255, 217, 232); padding: 0.5em 1em 0.5em 1.2em; margin: 1em 0; border-radius: 5px; max-width:50ch;">
  <h2> Question. </h2>
  <strong>How clear was this description to you? Do you think that you could implement this exactly based off of this description? Or do you think you'd need access to the source code?</strong>
</div>

### Applications of Attention in this Paper

In this transformer architecture, attention is used in several different ways. If you have requirements where a model needs to differentially choose to attend to different parts of the input, you can use this architecture.

<img src="assets/paper-09-attention-applications.png" alt="The complete three bullets in Section 3.2.3" style="max-height:500px; width:auto; display:block; margin:auto;">

### Skipping Ahead a Section: Why Self-Attention?

<img src="assets/paper-11a-self-attention-table.png" alt="Table 1 from the paper" style="max-height:350px; width:auto;display:block; margin:auto;">

The authors describe several computational concerns, notably those that impact training and inference times. The proposed implementation is distinct from the RNN, LSTM, and CNN implementations at the time.

<img src="assets/paper-11b-self-attention-value-pt1.png" alt="Highlighted Section 4 contribution claims page 1" style="max-height:575px; width:auto;display:block; margin:auto;">

<img src="assets/paper-11b-self-attention-value-pt2.png" alt="Highlighted Section 4 contribution claims page 2" style="max-height:575px; width:auto;display:block; margin:auto;">

## Other Transformer Components

### Feed-Forward Networks and Embeddings

The model applies the same fully connected feed-forward network independently at each token position; attention is the component that mixes information across positions. Learned token embeddings are combined with positional encodings before entering the stacks.

<img src="assets/paper-10a-feed-forward-and-embeddings.png" alt="Sections 3.3 and 3.4" style="max-height:600px; width:auto; display:block; margin:auto;">

### Positional Encoding
Without recurrence or convolution, position must be supplied another way. The paper uses sinusoidal encodings; GPT-2 uses learned position embeddings.

<img src="assets/paper-10b-positional-encoding.png" alt="Complete Section 3.5" style="max-height:720px; width:auto; display:block; margin:auto;">

In [ ]:
print("Token embeddings:", gpt2.wte)
print("Position embeddings:", gpt2.wpe)
print("MLP:", gpt2.h[0].mlp)
print("Final normalization:", gpt2.ln_f)

## Another Brief Diversion: How does inference actually work?

`text → tokens → token IDs → hidden states → logits → next token`





<img src="assets/full-nlp-pipeline.svg" alt="Tokenizer, transformer model, and postprocessing pipeline" style="max-height:430px; width:auto;">

In [ ]:
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

We need to first **tokenize** the text - that is, turning the text into a sequence of tokens and numbers.

In [ ]:
text = "Attention replaces recurrence"
inputs = tokenizer(text, return_tensors="pt")
inputs

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])

print("Tokens:", tokens)
print("Token IDs:", inputs.input_ids[0].tolist())
print("Shape:", inputs.input_ids.shape)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

outputs.keys()

In [ ]:
print("Number of hidden-state tensors:", len(outputs.hidden_states))
print("Embedding output:", outputs.hidden_states[0].shape)
print("After the first block:", outputs.hidden_states[1].shape)
print("Final hidden state:", outputs.hidden_states[-1].shape)
print("Logits:", outputs.logits.shape)

In [ ]:
next_token_logits = outputs.logits[0, -1]
probabilities = torch.softmax(next_token_logits, dim=-1)
top = torch.topk(probabilities, k=5)

for token_id, probability in zip(top.indices, top.values):
    print(f"{tokenizer.decode(token_id):>12}  {probability:.3f}")

## Training, Results, and Conclusion

**Main takeaway**: This paper presents the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention.

The following sections will be covered in later units:
- **Training:** data, batching, hardware, optimization, and regularization
- **Results:** translation quality, training cost, and decoding with beam search
- **Conclusion:** what the authors believed they established and what remained open

# Next Time

WOW was the most important section somewhat opaque. What about reproducibility, or even simply correctly implementing the model? **Formal Methods for Transformers** helps to solve this problem, and we'll see it in the next class!